# 01 · DeepFilterNet2/3 — Denoising

Notebook de inferencia con DeepFilterNet (paquete pip propio, no `transformers`).
Categoría: **denoising**. Se compara contra el baseline clásico de spectral gating.

Requiere haber ejecutado antes `00_Setup_Base.ipynb` (Drive montado, HF_HOME configurado,
utils guardadas en `utils/`, audio del tutor subido a `audio_samples/`).


## 1. Montar Drive y configurar entorno

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'


Mounted at /content/drive


## 2. Instalar dependencias comunes

Los paquetes de `pip` no persisten entre sesiones de Colab (solo los archivos en Drive sí),
así que hay que reinstalar estas dependencias en cada sesión nueva.


In [3]:
!pip install -q librosa soundfile scipy pesq pystoi speechmos onnxruntime matplotlib pandas


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.7 MB/s eta 0:00:00


## 3. Importar utilidades comunes (desde utils/ en Drive)

In [4]:
import sys
sys.path.append(f'{PROJECT_ROOT}/utils')

from audio_utils_funcionescomunes import cargar_audio, guardar_audio, resamplear, normalizar_pico
from audio_utils_memoria_GPU import liberar_memoria_gpu
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_baselines_clasicos import baseline_denoising_spectral_gating


## 4. Instalar dependencias específicas de DeepFilterNet

DeepFilterNet se instala como paquete pip propio (`deepfilternet`), no vía `transformers`.
Su dependencia nativa `DeepFilterLib` está escrita en Rust y no tiene wheel precompilado
para el entorno de Colab (Python 3.12), así que hace falta instalar el compilador de Rust
antes de poder instalar `deepfilternet`. Esto tarda 1-2 minutos y hay que repetirlo en cada
sesión nueva.

**No fijamos `numpy<2.0` a mano**: el propio `deepfilternet` resuelve la versión de numpy
que necesita (1.26.x) al instalarse. Forzarlo antes solo hace que se sobreescriba igual.

**Paso crítico — reiniciar el kernel después de instalar:** una vez termine la celda
siguiente, hay que reiniciar el entorno de ejecución (*Entorno de ejecución → Reiniciar
sesión*) ANTES de continuar con el resto del notebook. Esto es obligatorio: si no se
reinicia, numpy puede quedar cacheado en memoria en la versión que tenía el kernel *antes*
de instalar `deepfilternet` (2.x por defecto en Colab), aunque en disco ya esté en 1.26.x —
y ese desajuste entre memoria y disco rompe `init_df()` con un error de compatibilidad
binaria (ABI) de Rust. No hace falta reinstalar nada tras el reinicio, solo reiniciar el
kernel y continuar ejecutando las celdas siguientes con normalidad.


In [5]:
import os

# Instalamos el toolchain de Rust (necesario para compilar DeepFilterLib)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!rustc --version


info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-09 for version 1.97.0 (2d8144b78 2026-07-07)
info: downloading 6 components
        cargo downloading [##             ]   10.63 MiB (82.78 MiB/s, ETA: 0s)
        cargo downloading [########       ]   10.63 MiB (76.04 MiB/s, ETA: 0s)
        cargo downloading [#############  ]   10.63 MiB (78.41 MiB/s, ETA: 0s)
        cargo downloading [###############]   10.63 MiB (77.80 MiB/s, ETA: 0s)
        cargo pending installation            10.63 MiB
        cargo pending installation            10.63 MiB
       clippy unpacking   [#####         

In [6]:
# Instalamos deepfilternet. Los avisos de "pip's dependency resolver..." sobre paquetes
# preinstalados en Colab (jax, opencv, xarray, etc.) son ruido: no los usamos en este
# notebook y no afectan a la instalación. Lo que importa es que "Building wheel for
# deepfilterlib" termine en "done" sin errores por encima.
!pip install -q --no-cache-dir deepfilternet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 41.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 346.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.2/113.2 kB 396.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 345.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 193.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 177.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
db-dtypes 1.7.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
jax 0.7.2 requires 

### >>> REINICIA EL KERNEL AHORA <<<

*Entorno de ejecución → Reiniciar sesión.* Después de reiniciar, continúa por la celda
siguiente (no hace falta repetir el montaje de Drive ni las instalaciones anteriores,
pero sí las celdas de imports y configuración, ya que el reinicio borra el estado en
memoria, no los paquetes instalados en disco).


## 5. Tras el reinicio: volver a montar Drive e importar utilidades

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'

import sys
sys.path.append(f'{PROJECT_ROOT}/utils')

from audio_utils_funcionescomunes import cargar_audio, guardar_audio, resamplear, normalizar_pico
from audio_utils_memoria_GPU import liberar_memoria_gpu
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_baselines_clasicos import baseline_denoising_spectral_gating

import numpy
print('Version de numpy activa (debe ser 1.26.x):', numpy.__version__)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Version de numpy activa (debe ser 1.26.x): 1.26.4


## 6. Cargar el audio de prueba

Muestra los audios disponibles en `audio_samples/` y pide cuál usar.


In [2]:
carpeta_audios = f'{PROJECT_ROOT}/audio_samples'

print('Audios disponibles en audio_samples/:')
for archivo in os.listdir(carpeta_audios):
    print(f'  - {archivo}')

nombre_audio = input('\nIntroduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): ').strip()
RUTA_AUDIO_ORIGINAL = f'{carpeta_audios}/{nombre_audio}'

if not os.path.isfile(RUTA_AUDIO_ORIGINAL):
    raise FileNotFoundError(f'No se ha encontrado el archivo: {RUTA_AUDIO_ORIGINAL}')

# Nombre base sin extensión, para usarlo luego al nombrar los archivos de salida
NOMBRE_BASE = os.path.splitext(nombre_audio)[0]

# Cargamos con su sample rate original; DeepFilterNet resamplea internamente si hace falta
audio_original, sr_original = cargar_audio(RUTA_AUDIO_ORIGINAL, sr_objetivo=None, forzar_mono=True)
print(f'\nAudio cargado: {len(audio_original)/sr_original:.1f} s, {sr_original} Hz')


Audios disponibles en audio_samples/:
  - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): AUDIO_REVERB_ALBIOL_TFG.wav

Audio cargado: 88.0 s, 44100 Hz


## 7. Cargar el modelo DeepFilterNet

`init_df()` descarga (o recupera de caché) el modelo preentrenado y devuelve el modelo,
el estado (`df_state`, incluye el sample rate esperado) y la config. Por defecto carga
DeepFilterNet3.

**Nota de compatibilidad:** `deepfilternet` usa `torchaudio.backend.common.AudioMetaData`
y `torchaudio.info()`, ambos eliminados en `torchaudio` >=2.9 (parte de su migración a
`torchcodec`). El shim de abajo los reconstruye sin necesidad de downgradear `torch`/
`torchaudio`.


In [3]:
import types
import torchaudio
import soundfile as sf

# 1. Shim de la clase AudioMetaData (torchaudio.backend.common), eliminada en >=2.9
if not hasattr(torchaudio, 'backend'):
    backend_module = types.ModuleType('torchaudio.backend')
    common_module = types.ModuleType('torchaudio.backend.common')

    class AudioMetaData:
        """Stub de compatibilidad: la clase real fue eliminada en
        torchaudio >=2.9. Solo se usa como type hint interno en deepfilternet,
        no afecta a la inferencia."""
        def __init__(self, sample_rate=None, num_frames=None, num_channels=None,
                     bits_per_sample=None, encoding=None):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding

    common_module.AudioMetaData = AudioMetaData
    backend_module.common = common_module
    sys.modules['torchaudio.backend'] = backend_module
    sys.modules['torchaudio.backend.common'] = common_module
else:
    from torchaudio.backend.common import AudioMetaData

# 2. Shim de torchaudio.info(), eliminada por completo en torchaudio >=2.9
#    (a diferencia de load()/save(), que siguen existiendo como alias a
#    *_with_torchcodec()). df/io.py solo necesita el atributo .sample_rate,
#    asi que lo reconstruimos con soundfile.
if not hasattr(torchaudio, 'info'):
    def _info_shim(file, **kwargs):
        datos = sf.info(str(file))
        return AudioMetaData(
            sample_rate=datos.samplerate,
            num_frames=datos.frames,
            num_channels=datos.channels,
            bits_per_sample=0,
            encoding=datos.subtype,
        )
    torchaudio.info = _info_shim

print('Shims de compatibilidad torchaudio aplicados (si hacia falta).')


Shims de compatibilidad torchaudio aplicados (si hacia falta).


In [4]:
from df.enhance import enhance, init_df, load_audio, save_audio

modelo_dfn, df_state, _ = init_df()  # por defecto: DeepFilterNet3
sr_modelo = df_state.sr()
print(f'Modelo cargado. Sample rate esperado por el modelo: {sr_modelo} Hz')


2026-07-11 15:24:19 | INFO     | DF | Running on torch 2.11.0+cu128
2026-07-11 15:24:19 | INFO     | DF | Running on host 9f1448b87372
2026-07-11 15:24:19 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-07-11 15:24:19 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2026-07-11 15:24:19 | INFO     | DF | Initializing model `deepfilternet3`
2026-07-11 15:24:20 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-07-11 15:24:20 | INFO     | DF | Running on device cuda:0
2026-07-11 15:24:20 | INFO     | DF | Model loaded
Modelo cargado. Sample rate esperado por el modelo: 48000 Hz


## 8. Inferencia sobre el audio

In [5]:
# DeepFilterNet tiene su propia función load_audio que ya resamplea al sr correcto del modelo
audio_entrada, _ = load_audio(RUTA_AUDIO_ORIGINAL, sr=sr_modelo)

audio_mejorado = enhance(modelo_dfn, df_state, audio_entrada)

RUTA_SALIDA = f'{PROJECT_ROOT}/outputs/deepfilternet/{NOMBRE_BASE}_denoised.wav'
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)
save_audio(RUTA_SALIDA, audio_mejorado, sr_modelo)
print(f'Audio mejorado guardado en: {RUTA_SALIDA}')


2026-07-11 15:24:30 | WARNING  | DF | Audio sampling rate does not match model sampling rate (44100, 48000). Resampling...
/usr/local/lib/python3.12/dist-packages/df/io.py:106: UserWarning: "sinc_interpolation" resampling method name is being deprecated and replaced by "sinc_interp_hann" in the next release. The default behavior remains unchanged.
  return ta_resample(audio, orig_sr, new_sr, **params)


Audio mejorado guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/deepfilternet/AUDIO_REVERB_ALBIOL_TFG_denoised.wav


## 9. Baseline clásico (spectral gating) sobre el mismo audio

In [6]:
audio_baseline = baseline_denoising_spectral_gating(audio_original, sr_original)

RUTA_BASELINE = f'{PROJECT_ROOT}/outputs/baseline_denoising/{NOMBRE_BASE}_baseline.wav'
os.makedirs(os.path.dirname(RUTA_BASELINE), exist_ok=True)
guardar_audio(RUTA_BASELINE, audio_baseline, sr_original)
print(f'Audio baseline guardado en: {RUTA_BASELINE}')


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_denoising/AUDIO_REVERB_ALBIOL_TFG_baseline.wav
Audio baseline guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_denoising/AUDIO_REVERB_ALBIOL_TFG_baseline.wav


## 10. Calcular métricas (DNSMOS)

Como no hay audio limpio de referencia para la grabación del tutor, usamos DNSMOS
(métrica no-intrusiva) sobre: audio original, salida de DeepFilterNet, y salida del
baseline. `calcular_dnsmos` resamplea a 16 kHz y reduce a mono automáticamente si hace
falta (el audio de partida es estéreo).


In [7]:
import numpy as np

audio_mejorado_arr = np.array(audio_mejorado).squeeze()

dnsmos_original  = calcular_dnsmos(audio_original, sr_original)
dnsmos_dfn       = calcular_dnsmos(audio_mejorado_arr, sr_modelo)
dnsmos_baseline  = calcular_dnsmos(audio_baseline, sr_original)

print('DNSMOS — Audio original:      ', dnsmos_original)
print('DNSMOS — DeepFilterNet:       ', dnsmos_dfn)
print('DNSMOS — Baseline (no-IA):    ', dnsmos_baseline)


DNSMOS — Audio original:       {'ovrl_mos': 1.3853487473266228, 'sig_mos': 1.5779884955663657, 'bak_mos': 1.790054065636629, 'p808_mos': 2.426125}
DNSMOS — DeepFilterNet:        {'ovrl_mos': 2.0899550470148296, 'sig_mos': 2.308938298301112, 'bak_mos': 3.747231940518648, 'p808_mos': 2.4446242}
DNSMOS — Baseline (no-IA):     {'ovrl_mos': 1.4003767401759506, 'sig_mos': 1.578206735172742, 'bak_mos': 1.8430690455049095, 'p808_mos': 2.431142}


## 11. Tabla resumen de la comparativa

In [8]:
import pandas as pd

resumen = pd.DataFrame([
    {'Version': 'Original (degradado)', 'OVRL': dnsmos_original.get('ovrl_mos'), 'SIG': dnsmos_original.get('sig_mos'), 'BAK': dnsmos_original.get('bak_mos')},
    {'Version': 'DeepFilterNet (IA)',   'OVRL': dnsmos_dfn.get('ovrl_mos'),      'SIG': dnsmos_dfn.get('sig_mos'),      'BAK': dnsmos_dfn.get('bak_mos')},
    {'Version': 'Baseline (no-IA)',     'OVRL': dnsmos_baseline.get('ovrl_mos'), 'SIG': dnsmos_baseline.get('sig_mos'), 'BAK': dnsmos_baseline.get('bak_mos')},
])
resumen


,Version,OVRL,SIG,BAK
0,Original (degradado),1.385349,1.577988,1.790054
1,DeepFilterNet (IA),2.089955,2.308938,3.747232
2,Baseline (no-IA),1.400377,1.578207,1.843069


## 12. Liberar memoria GPU

In [9]:
liberar_memoria_gpu(modelo_dfn)


Aviso: sin nombre_variable, solo se libera la referencia local a esta función. Si el modelo sigue asignado a una variable en el notebook (ej. modelo_dfn), la GPU no se liberará del todo. Llama a esta función como liberar_memoria_gpu(modelo_dfn, 'modelo_dfn') para liberarla de verdad.
Memoria GPU liberada. Uso actual: 0.01 GB


## Próximo paso

Con DeepFilterNet ya probado y comparado contra su baseline, el siguiente notebook es
`02_HTDemucs.ipynb` (separación de fuentes), siguiendo el orden acordado en la Fase 2.
